# ETL Pipeline — Dacon E-Commerce Dataset

## 분석 흐름
1. **Data Overview** — 각 테이블 기본 구조 파악  
2. **ETL** — 정제 → 조인 → 파생변수 생성 → MySQL 저장

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv
import os

load_dotenv()  # 프로젝트 루트의 .env 자동 탐색
DATA_DIR = os.getenv('DATA_DIR')


---
## 1. Data Overview

- **데이터 출처**: https://dacon.io/competitions/official/236222/data
- **분석 기간**: 2019-01-01 ~ 2019-12-31

| 파일 | 설명 |
|------|------|
| `Onlinesales_info.csv` | 거래 내역 (메인 팩트 테이블) |
| `Customer_info.csv` | 고객 인구통계 정보 |
| `Discount_info.csv` | 월별 카테고리 쿠폰 정보 |
| `Marketing_info.csv` | 일별 마케팅 비용 |
| `Tax_info.csv` | 카테고리별 GST 세율 |

### 1. Onlinesales_info.csv
- 고객별 거래 내역. 하나의 거래ID에 여러 상품(행)이 포함될 수 있는 장바구니 구조

### 컬럼 별 정보
- **고객ID**: 고객 고유 식별자
- **거래ID**: 거래 고유 식별자 (1거래 = 여러 행 가능)
- **거래날짜**: 거래 발생일
- **제품ID**: 상품 고유 식별자
- **제품카테고리**: 상품 카테고리 (20종)
- **수량**: 구매 수량
- **평균금액**: 상품 단가
- **배송료**: 배송비
- **쿠폰상태**: 쿠폰 적용 상태 (Used / Clicked / Not Used)

In [2]:
sales = pd.read_csv(f'{DATA_DIR}/Onlinesales_info.csv')
sales.head()

,고객ID,거래ID,거래날짜,제품ID,제품카테고리,수량,평균금액,배송료,쿠폰상태
0,USER_1358,Transaction_0000,2019-01-01,Product_0981,Nest-USA,1,153.71,6.5,Used
1,USER_1358,Transaction_0001,2019-01-01,Product_0981,Nest-USA,1,153.71,6.5,Used
2,USER_1358,Transaction_0002,2019-01-01,Product_0904,Office,1,2.05,6.5,Used
3,USER_1358,Transaction_0003,2019-01-01,Product_0203,Apparel,5,17.53,6.5,Not Used
4,USER_1358,Transaction_0003,2019-01-01,Product_0848,Bags,1,16.50,6.5,Used


In [3]:
sales.shape

(52924, 9)

In [4]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52924 entries, 0 to 52923
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   고객ID    52924 non-null  object 
 1   거래ID    52924 non-null  object 
 2   거래날짜    52924 non-null  object 
 3   제품ID    52924 non-null  object 
 4   제품카테고리  52924 non-null  object 
 5   수량      52924 non-null  int64  
 6   평균금액    52924 non-null  float64
 7   배송료     52924 non-null  float64
 8   쿠폰상태    52924 non-null  object 
dtypes: float64(2), int64(1), object(6)
memory usage: 3.6+ MB


### 2. Customer_info.csv
- 고객의 인구통계 정보. 고객ID 기준으로 Onlinesales와 1:1 조인

### 컬럼 별 정보
- **고객ID**: 고객 고유 식별자
- **성별**: 고객 성별
- **고객지역**: 고객 거주 지역
- **가입기간**: 가입 후 경과 개월 수

In [5]:
customer = pd.read_csv(f'{DATA_DIR}/Customer_info.csv')
customer.head()

,고객ID,성별,고객지역,가입기간
0,USER_1358,남,Chicago,12
1,USER_0190,남,California,43
2,USER_0066,남,Chicago,33
3,USER_0345,여,California,30
4,USER_0683,남,California,49


In [6]:
customer.shape

(1468, 4)

In [7]:
customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1468 entries, 0 to 1467
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   고객ID    1468 non-null   object
 1   성별      1468 non-null   object
 2   고객지역    1468 non-null   object
 3   가입기간    1468 non-null   int64 
dtypes: int64(1), object(3)
memory usage: 46.0+ KB


### 3. Discount_info.csv
- 월별 × 카테고리별 쿠폰 코드 및 할인율. 거래날짜의 월 + 제품카테고리로 조인
- 원본에 `Notebooks`와 `Notebooks & Journals`가 별개 항목으로 존재 (이름 변환 없이 사용)
- `Backpacks`, `More Bags`, `Fun`, `Google` 카테고리는 쿠폰 데이터 없음

### 컬럼 별 정보
- **월**: 적용 월 (Jan ~ Dec)
- **제품카테고리**: 쿠폰 적용 카테고리
- **쿠폰코드**: 쿠폰 코드명
- **할인율**: 할인율 (10 / 20 / 30 %)

In [8]:
discount = pd.read_csv(f'{DATA_DIR}/Discount_info.csv')
discount.head()

,월,제품카테고리,쿠폰코드,할인율
0,Jan,Apparel,SALE10,10
1,Feb,Apparel,SALE20,20
2,Mar,Apparel,SALE30,30
3,Jan,Nest-USA,ELEC10,10
4,Feb,Nest-USA,ELEC20,20


In [9]:
discount.shape

(204, 4)

In [10]:
discount.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 204 entries, 0 to 203
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   월       204 non-null    object
 1   제품카테고리  204 non-null    object
 2   쿠폰코드    204 non-null    object
 3   할인율     204 non-null    int64 
dtypes: int64(1), object(3)
memory usage: 6.5+ KB


### 4. Marketing_info.csv
- 날짜별 마케팅 비용. 거래날짜 기준으로 조인하여 당일 마케팅 비용을 거래에 부착

### 컬럼 별 정보
- **날짜**: 마케팅 비용 집계일
- **오프라인비용**: 당일 오프라인 마케팅 집행 비용
- **온라인비용**: 당일 온라인 마케팅 집행 비용

In [11]:
marketing = pd.read_csv(f'{DATA_DIR}/Marketing_info.csv')
marketing.head()

,날짜,오프라인비용,온라인비용
0,2019-01-01,4500,2424.50
1,2019-01-02,4500,3480.36
2,2019-01-03,4500,1576.38
3,2019-01-04,4500,2928.55
4,2019-01-05,4500,4055.30


In [12]:
marketing.shape

(365, 3)

In [13]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   날짜      365 non-null    object 
 1   오프라인비용  365 non-null    int64  
 2   온라인비용   365 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 8.7+ KB


### 5. Tax_info.csv
- 카테고리별 GST(Goods and Services Tax) 세율. 제품카테고리 기준으로 조인

### 컬럼 별 정보
- **제품카테고리**: 상품 카테고리명
- **GST**: 세율 (0.05 ~ 0.18)

In [14]:
tax = pd.read_csv(f'{DATA_DIR}/Tax_info.csv')
tax.head()

,제품카테고리,GST
0,Nest-USA,0.10
1,Office,0.10
2,Apparel,0.18
3,Bags,0.18
4,Drinkware,0.18


In [15]:
tax.shape

(20, 2)

In [16]:
tax.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   제품카테고리  20 non-null     object 
 1   GST     20 non-null     float64
dtypes: float64(1), object(1)
memory usage: 448.0+ bytes


---
## 2. ETL

### 단계별 흐름

| 단계 | 작업 | 추가 컬럼 |
|------|------|---------|
| 2-1 | Sales 기본 정제 | `연도`, `월`, `총매출` |
| 2-2 | Discount 정제 | `월_num` (조인용 임시) |
| 2-3 | Tax 조인 | `gst` |
| 2-4 | Discount 조인 + 파생변수 | `쿠폰코드`, `할인율`, `할인적용여부`, `할인후금액`, `세후금액` |
| 2-5 | Customer 조인 | `성별`, `고객지역`, `가입기간` |
| 2-6 | Marketing 조인 | `오프라인비용`, `온라인비용` |
| 2-7 | 총 거래금액 계산 | `최종결제금액`, `거래_총금액` |

### 파생변수 계산식
| 컬럼 | 계산식 |
|------|--------|
| `총매출` | `평균금액 × 수량` |
| `할인후금액` | `총매출 × (1 - 할인율/100)` (쿠폰 Used일 때만, 나머지는 총매출 그대로) |
| `세후금액` | `할인후금액 × (1 + GST)` |
| `최종결제금액` | `세후금액 + 배송료` |
| `거래_총금액` | 거래ID별 `최종결제금액` 합산 |

In [17]:
# 2-1. Sales 기본 정제
df = sales.copy()

# Fun, Google 카테고리 분석 대상 제외
# - Discount 테이블에 해당 카테고리 쿠폰 정보가 없고
# - More Bags / Backpacks와 달리 도메인 지식으로도 상위 카테고리를 특정할 수 없어 할인율 추정 불가
# - 전체 52,924건 중 두 카테고리 합산 약 250건 미만(0.5% 미만)으로 분석 영향 미미
df = df[~df['제품카테고리'].isin(['Fun', 'Google'])]

df['거래날짜'] = pd.to_datetime(df['거래날짜'])
df['연도'] = df['거래날짜'].dt.year
df['월'] = df['거래날짜'].dt.month
df['총매출'] = (df['평균금액'] * df['수량']).round(2)

In [18]:
df[['고객ID','거래ID','거래날짜','제품카테고리','수량','평균금액','총매출']].head()

,고객ID,거래ID,거래날짜,제품카테고리,수량,평균금액,총매출
0,USER_1358,Transaction_0000,2019-01-01,Nest-USA,1,153.71,153.71
1,USER_1358,Transaction_0001,2019-01-01,Nest-USA,1,153.71,153.71
2,USER_1358,Transaction_0002,2019-01-01,Office,1,2.05,2.05
3,USER_1358,Transaction_0003,2019-01-01,Apparel,5,17.53,87.65
4,USER_1358,Transaction_0003,2019-01-01,Bags,1,16.50,16.50


In [19]:
# 2-2. Discount 정제
# 원본에 'Notebooks'와 'Notebooks & Journals'가 별개 항목으로 존재.
# Sales에는 'Notebooks & Journals'만 있으므로 이름 변환 없이 그대로 조인.
discount_clean = discount.copy()
month_map = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
             'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}
discount_clean['월_num'] = discount_clean['월'].map(month_map)

In [20]:
discount_clean.head()

,월,제품카테고리,쿠폰코드,할인율,월_num
0,Jan,Apparel,SALE10,10,1
1,Feb,Apparel,SALE20,20,2
2,Mar,Apparel,SALE30,30,3
3,Jan,Nest-USA,ELEC10,10,1
4,Feb,Nest-USA,ELEC20,20,2


In [21]:
# 2-3. Tax 조인
# 세후금액은 할인 적용 이후에 계산해야 하므로 여기선 gst 컬럼만 붙임
df = df.merge(tax.rename(columns={'GST': 'gst'}), on='제품카테고리', how='left')
assert df['gst'].isnull().sum() == 0, 'gst null 존재 — Tax 테이블 카테고리 미매핑 확인 필요'

In [22]:
df[['제품카테고리','총매출','gst']].head()

,제품카테고리,총매출,gst
0,Nest-USA,153.71,0.10
1,Nest-USA,153.71,0.10
2,Office,2.05,0.10
3,Apparel,87.65,0.18
4,Bags,16.50,0.18


In [23]:
# 2-4. Discount 조인 (월 + 카테고리 기준)
# More Bags / Backpacks → Bags로 임시 매핑하여 조인
# (두 카테고리는 Bags의 하위 카테고리로 동일 쿠폰 적용)
cat_map = {'More Bags': 'Bags', 'Backpacks': 'Bags'}
df['_join_cat'] = df['제품카테고리'].replace(cat_map)

discount_join = discount_clean[['월_num','제품카테고리','쿠폰코드','할인율']].rename(columns={'제품카테고리': '_discount_cat'})
df = df.merge(discount_join, left_on=['월','_join_cat'], right_on=['월_num','_discount_cat'], how='left')
df['할인적용여부'] = (df['쿠폰상태'] == 'Used').astype(int)
df.drop(columns=['_join_cat', '_discount_cat', '월_num'], inplace=True, errors='ignore')

df['할인율'] = df['할인율'].fillna(0)
df['쿠폰코드'] = df['쿠폰코드'].fillna('없음')
assert df['할인율'].isnull().sum() == 0, '할인율 null 잔존 — 카테고리 매핑 확인 필요'

# 할인 적용 후 세금 계산
# 할인후금액: 쿠폰 Used일 때만 할인율 반영
df['할인후금액'] = (df['총매출'] * (1 - df['할인율'] / 100 * df['할인적용여부'])).round(2)
df['세후금액'] = (df['할인후금액'] * (1 + df['gst'])).round(2)

In [24]:
df[['제품카테고리','총매출','할인율','할인적용여부','할인후금액','gst','세후금액']].head()

,제품카테고리,총매출,할인율,할인적용여부,할인후금액,gst,세후금액
0,Nest-USA,153.71,10,1,138.34,0.10,152.17
1,Nest-USA,153.71,10,1,138.34,0.10,152.17
2,Office,2.05,10,1,1.84,0.10,2.02
3,Apparel,87.65,10,0,87.65,0.18,103.43
4,Bags,16.50,10,1,14.85,0.18,17.52


In [25]:
# 2-5. Customer 조인
df = df.merge(customer, on='고객ID', how='left')
assert df['성별'].isnull().sum() == 0, '성별 null 존재 — Customer 테이블 고객ID 미매핑 확인 필요'

In [26]:
df[['고객ID','성별','고객지역','가입기간']].head()

,고객ID,성별,고객지역,가입기간
0,USER_1358,남,Chicago,12
1,USER_1358,남,Chicago,12
2,USER_1358,남,Chicago,12
3,USER_1358,남,Chicago,12
4,USER_1358,남,Chicago,12


In [27]:
# 2-6. Marketing 조인 (날짜 기준)
marketing_clean = marketing.copy()
marketing_clean['날짜'] = pd.to_datetime(marketing_clean['날짜'])
df = df.merge(marketing_clean, left_on='거래날짜', right_on='날짜', how='left')
df.drop(columns=['날짜'], inplace=True, errors='ignore')
assert df['오프라인비용'].isnull().sum() == 0, '오프라인비용 null 존재 — Marketing 날짜 미매핑 확인 필요'

In [28]:
df[['거래날짜','오프라인비용','온라인비용']].head()

,거래날짜,오프라인비용,온라인비용
0,2019-01-01,4500,2424.5
1,2019-01-01,4500,2424.5
2,2019-01-01,4500,2424.5
3,2019-01-01,4500,2424.5
4,2019-01-01,4500,2424.5


In [29]:
# 2-7. 총 거래금액 계산
# 최종결제금액: 세후금액 + 배송료 (행 단위 실결제금액)
df['최종결제금액'] = (df['세후금액'] + df['배송료']).round(2)

# 거래_총금액: 거래ID 기준 합산 (장바구니 전체 결제금액)
df['거래_총금액'] = df.groupby('거래ID')['최종결제금액'].transform('sum').round(2)

In [30]:
df[['거래ID','제품카테고리','세후금액','배송료','최종결제금액','거래_총금액']].head()

,거래ID,제품카테고리,세후금액,배송료,최종결제금액,거래_총금액
0,Transaction_0000,Nest-USA,152.17,6.5,158.67,158.67
1,Transaction_0001,Nest-USA,152.17,6.5,158.67,158.67
2,Transaction_0002,Office,2.02,6.5,8.52,8.52
3,Transaction_0003,Apparel,103.43,6.5,109.93,833.50
4,Transaction_0003,Bags,17.52,6.5,24.02,833.50


In [31]:
# 최종 확인
print(f'종합 테이블 shape: {df.shape}')
remaining = df.isnull().sum()
print(f'null: {remaining[remaining > 0].to_string() if remaining.sum() > 0 else "없음"}')
df.head()
print(df.info())

종합 테이블 shape: (52659, 25)
null: 없음
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52659 entries, 0 to 52658
Data columns (total 25 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   고객ID    52659 non-null  object        
 1   거래ID    52659 non-null  object        
 2   거래날짜    52659 non-null  datetime64[ns]
 3   제품ID    52659 non-null  object        
 4   제품카테고리  52659 non-null  object        
 5   수량      52659 non-null  int64         
 6   평균금액    52659 non-null  float64       
 7   배송료     52659 non-null  float64       
 8   쿠폰상태    52659 non-null  object        
 9   연도      52659 non-null  int32         
 10  월       52659 non-null  int32         
 11  총매출     52659 non-null  float64       
 12  gst     52659 non-null  float64       
 13  쿠폰코드    52659 non-null  object        
 14  할인율     52659 non-null  int64         
 15  할인적용여부  52659 non-null  int64         
 16  할인후금액   52659 non-null  float64       
 17  세후금액    52659 n

---
## 3. Load — MySQL 적재

| 테이블 | 내용 |
|--------|------|
| `onlinesales` | 거래 내역 원본 (Onlinesales_info.csv) |
| `orders_master` | 거래 + 고객 + 세금 + 할인 + 마케팅 조인 마스터 |
| `customers` | 고객 정보 원본 |
| `marketing` | 일별 마케팅 비용 |
| `tax` | 카테고리별 GST |
| `discount` | 월별 카테고리 쿠폰 |

In [32]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()  # 프로젝트 루트의 .env 자동 탐색

engine = create_engine(
    f"mysql+mysqlconnector://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)

print(f"{os.getenv('DB_HOST')} / {os.getenv('DB_NAME')}")


localhost / Dacon_Ecommerce


In [33]:
# orders_master: 조인 마스터 테이블
df.to_sql('orders_master', con=engine, if_exists='replace', index=False)

# onlinesales: 거래 내역 원본
sales.to_sql('onlinesales', con=engine, if_exists='replace', index=False)

# customers: 고객 인구통계 원본
customer.to_sql('customers', con=engine, if_exists='replace', index=False)

# marketing: 일별 마케팅 비용
marketing_clean.to_sql('marketing', con=engine, if_exists='replace', index=False)

# tax: 카테고리별 GST 세율
tax.to_sql('tax', con=engine, if_exists='replace', index=False)

# discount: 월별 카테고리 쿠폰
discount_clean.to_sql('discount', con=engine, if_exists='replace', index=False)

204

In [34]:
# 적재 검증
for tbl in ['onlinesales', 'orders_master', 'customers', 'marketing', 'tax', 'discount']:
    cnt = pd.read_sql(f'SELECT COUNT(*) as n FROM {tbl}', engine).iloc[0, 0]
    print(f'{tbl}: {cnt:,}행')

# orders_master 날짜 범위 확인
date_range = pd.read_sql(
    'SELECT MIN(거래날짜) as min_dt, MAX(거래날짜) as max_dt FROM orders_master', engine
).iloc[0]
print(f"\norders_master 날짜 범위: {date_range['min_dt']} ~ {date_range['max_dt']}")

# orders_master 컬럼별 null 비율
total = pd.read_sql('SELECT COUNT(*) as n FROM orders_master', engine).iloc[0, 0]
null_counts = pd.read_sql(
    """
    SELECT
        SUM(거래날짜 IS NULL) AS 거래날짜,
        SUM(제품카테고리 IS NULL) AS 제품카테고리,
        SUM(총매출 IS NULL) AS 총매출,
        SUM(할인율 IS NULL) AS 할인율,
        SUM(세후금액 IS NULL) AS 세후금액,
        SUM(고객ID IS NULL) AS 고객ID,
        SUM(성별 IS NULL) AS 성별,
        SUM(오프라인비용 IS NULL) AS 오프라인비용
    FROM orders_master""",
    engine
).iloc[0]
print("orders_master null 비율:")
for col, cnt in null_counts.items():
    pct = cnt / total * 100
    flag = '⚠' if pct > 0 else ''
    print(f'-{col}: {pct:.2f}%{flag}')


onlinesales: 52,924행
orders_master: 52,659행
customers: 1,468행
marketing: 365행
tax: 20행
discount: 204행

orders_master 날짜 범위: 2019-01-01 00:00:00 ~ 2019-12-31 00:00:00
orders_master null 비율:
-거래날짜: 0.00%
-제품카테고리: 0.00%
-총매출: 0.00%
-할인율: 0.00%
-세후금액: 0.00%
-고객ID: 0.00%
-성별: 0.00%
-오프라인비용: 0.00%


In [35]:
# orders_master 컬럼 목록
for c in df.columns:
    print(f'{c}: {df[c].dtype}')

고객ID: object
거래ID: object
거래날짜: datetime64[ns]
제품ID: object
제품카테고리: object
수량: int64
평균금액: float64
배송료: float64
쿠폰상태: object
연도: int32
월: int32
총매출: float64
gst: float64
쿠폰코드: object
할인율: int64
할인적용여부: int64
할인후금액: float64
세후금액: float64
성별: object
고객지역: object
가입기간: int64
오프라인비용: int64
온라인비용: float64
최종결제금액: float64
거래_총금액: float64
